# Atividade População 2010 e 2022

In [1]:
# No terminal, uma vez: python -m pip install openpyxl pandas
import pandas as pd
import openpyxl


In [2]:
path = r"H:\Python\analise_dados\CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

In [3]:
bruto = pd.read_excel(path, sheet_name="Municípios", header=None)
bruto.head()

,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833


In [4]:
dados = bruto.iloc[2:, 1:].copy()
dados.columns = [
    "uf", "cod_uf", "cod_municipio", "nome_municipio",
    "populacao_2010_sinopse", "populacao_2010_compatibilizada",
    "populacao_2022"
]
dados = dados.reset_index(drop=True)

for col in ["populacao_2010_sinopse", "populacao_2010_compatibilizada", "populacao_2022"]:
    dados[col] = pd.to_numeric(dados[col], errors="coerce")

dados = dados[dados["nome_municipio"].notna()].copy()
dados.head()


,uf,cod_uf,cod_municipio,nome_municipio,populacao_2010_sinopse,populacao_2010_compatibilizada,populacao_2022
0,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,NaN,NaN,NaN
1,RO,11,00015,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
2,RO,11,00023,Ariquemes,90353.0,90353.0,96833.0
3,RO,11,00031,Cabixi,6313.0,6313.0,5351.0
4,RO,11,00049,Cacoal,78574.0,78574.0,86887.0


## 1. População agregada por estado

In [5]:
populacao_estados = (
    dados.groupby("uf")[["populacao_2010_compatibilizada", "populacao_2022"]]
    .sum()
    .reset_index()
)
populacao_estados.head()


,uf,populacao_2010_compatibilizada,populacao_2022
0,AC,733559.0,830018.0
1,AL,3120887.0,3127683.0
2,AM,3483985.0,3941613.0
3,AP,669526.0,733759.0
4,BA,14017071.0,14141626.0


In [6]:
populacao_estados["crescimento_2010_2022"] = (
    populacao_estados["populacao_2022"]
    - populacao_estados["populacao_2010_compatibilizada"]
)

populacao_estados = populacao_estados.sort_values(
    "crescimento_2010_2022", ascending=False
).reset_index(drop=True)

populacao_estados

,uf,populacao_2010_compatibilizada,populacao_2022,crescimento_2010_2022
0,SP,41262199.0,44411238.0,3149039.0
1,SC,6248436.0,7610361.0,1361925.0
2,GO,6001789.0,7056495.0,1054706.0
3,PR,10444526.0,11444380.0,999854.0
4,MG,19597330.0,20539989.0,942659.0
5,MT,3035122.0,3658649.0,623527.0
6,PA,7581051.0,8120131.0,539080.0
7,AM,3483985.0,3941613.0,457628.0
8,CE,8451644.0,8794957.0,343313.0
9,ES,3514952.0,3833712.0,318760.0


In [7]:
populacao_estados.to_csv(
    r"populacao_por_estado_crescimento_2010_2022.csv",
    sep=";",
    index=False
)

## 2. População por município

In [8]:
populacao_municipios = dados[[
    "uf", "cod_uf", "cod_municipio", "nome_municipio",
    "populacao_2010_compatibilizada", "populacao_2022"
]].copy()

populacao_municipios["crescimento_2010_2022"] = (
    populacao_municipios["populacao_2022"]
    - populacao_municipios["populacao_2010_compatibilizada"]
)

populacao_municipios = populacao_municipios.sort_values(
    "crescimento_2010_2022", ascending=False
).reset_index(drop=True)

populacao_municipios.head(10)

,uf,cod_uf,cod_municipio,nome_municipio,populacao_2010_compatibilizada,populacao_2022,crescimento_2010_2022
0,AM,13,02603,Manaus,1802014.0,2063689.0,261675.0
1,DF,53,00108,Brasília,2572159.0,2817381.0,245222.0
2,SP,35,50308,São Paulo,11253503.0,11451999.0,198496.0
3,SP,35,52205,Sorocaba,586816.0,723682.0,136866.0
4,GO,52,08707,Goiânia,1301912.0,1437366.0,135454.0
5,RR,14,00100,Boa Vista,284313.0,413486.0,129173.0
6,SC,42,05407,Florianópolis,421240.0,537211.0,115971.0
7,PA,15,05536,Parauapebas,153908.0,267836.0,113928.0
8,MS,50,02704,Campo Grande,786774.0,898100.0,111326.0
9,PB,25,07507,João Pessoa,723515.0,833932.0,110417.0


In [9]:
populacao_municipios.to_csv(
    r"populacao_por_municipio_crescimento_2010_2022.csv",
    sep=";",
    index=False
)